# Rotirajući spremnik — predvidi, izračunaj, provjeri

**Poglavlje U04: relativno mirovanje pri rotaciji**

U cilindričnom spremniku profil slobodne površine dobivamo iz gradijenta
tlaka. Numerički ćemo integrirati radijalni gradijent i prikazati konvergenciju,
a očuvanje volumena poslužit će kao zasebna provjera.


## 1. Predvidi

1. Ako se kutna brzina udvostruči, koliko se puta mijenja razlika razina rub–os?
2. Ostaje li srednja visina fluida jednaka početnoj visini $h_0$?
3. Hoće li se pri rastu brzine prvo ogoliti dno u osi ili preliti rub?

Predviđanje provjeri tek nakon izvođenja sljedećih ćelija.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
RHO, G = 998.0, 9.81

def analiticki_profil(r, omega, R, h0):
    z_os = h0 - omega**2 * R**2 / (4*G)
    return z_os + omega**2 * np.asarray(r)**2 / (2*G)

omega, R, h0 = 7.0, 0.55, 0.70
z_os = analiticki_profil(0.0, omega, R, h0)
z_rub = analiticki_profil(R, omega, R, h0)
print(f"z_os = {z_os:.4f} m, z_rub = {z_rub:.4f} m")
print(f"razlika rub–os = {z_rub-z_os:.4f} m")


## 2. Izračunaj — numerička integracija gradijenta tlaka

U rotirajućem fluidu vrijedi

$$\frac{\partial p}{\partial r}=\rho\omega^2r,\qquad
\frac{\partial p}{\partial z}=-\rho g.$$

Na slobodnoj površini je $dp=0$, pa je $dz/dr=\omega^2r/g$.
Integriramo oba gradijenta lijevim pravokutnim pravilom. Budući da je metoda
prvog reda, prepolovljenje koraka trebalo bi približno prepoloviti pogrešku.


In [ ]:
def lijevi_integral_gradijenta(omega, R, n, rho=RHO):
    r = np.linspace(0.0, R, n+1)
    dr = R/n
    dp = np.sum(rho * omega**2 * r[:-1] * dr)
    dz = np.sum((omega**2/G) * r[:-1] * dr)
    return dp, dz

n_mreza = np.array([10, 20, 40, 80, 160, 320])
dp_ref = 0.5 * RHO * omega**2 * R**2
dz_ref = 0.5 * omega**2 * R**2 / G
dp_num = np.array([lijevi_integral_gradijenta(omega, R, int(n))[0]
                   for n in n_mreza])
dz_num = np.array([lijevi_integral_gradijenta(omega, R, int(n))[1]
                   for n in n_mreza])
err_dp = np.abs(dp_num - dp_ref)
err_dz = np.abs(dz_num - dz_ref)
p_red = np.log2(err_dp[:-1] / err_dp[1:])

print(" n    Δp_num [Pa]    pogreška [Pa]    opaženi red")
for i, n in enumerate(n_mreza):
    red = "--" if i == 0 else f"{p_red[i-1]:.3f}"
    print(f"{n:3d}  {dp_num[i]:12.4f}  {err_dp[i]:13.4f}  {red:>10s}")

fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.loglog(R/n_mreza, err_dp, "o-", label=r"pogreška $\Delta p$")
ax.loglog(R/n_mreza, err_dz*RHO*G, "s--",
          label=r"pogreška $\Delta z$ preračunata u Pa")
ax.set(xlabel=r"radijalni korak $\Delta r$ (m)", ylabel="apsolutna pogreška (Pa)",
       title="Konvergencija numeričke integracije")
ax.grid(ls=":", which="both", alpha=0.6)
ax.legend()
plt.show()


## 3. Provjeri — volumen, granični slučaj i red metode

Integracijom volumena paraboloida mora se vratiti početni volumen
$\pi R^2h_0$. Usto, za $\omega=0$ profil mora biti vodoravan.


In [ ]:
r_fino = np.linspace(0.0, R, 20001)
z_fino = analiticki_profil(r_fino, omega, R, h0)
# Trapezno integriranje V = integral 2*pi*r*z(r) dr.
integrand = 2*np.pi*r_fino*z_fino
V_num = np.sum(0.5*(integrand[:-1]+integrand[1:]) * np.diff(r_fino))
V_ref = np.pi * R**2 * h0
rel_V = abs(V_num - V_ref) / V_ref
print(f"relativni debalans volumena = {rel_V:.3e}")

assert np.all((p_red[-3:] > 0.98) & (p_red[-3:] < 1.02))
assert rel_V < 1e-8
assert np.allclose(analiticki_profil([0, R], 0.0, R, h0), h0)
print("PASS: prvi red konvergencije, volumen i mirujući granični slučaj.")


## Granica modela

Profil vrijedi nakon uspostave relativnog mirovanja i prije dodira površine
s dnom ili prelijevanja preko ruba. Za zadanu visinu stijenke oba događaja
treba računati zasebno; redoslijed ovisi o geometriji i početnom punjenju.
